# Programming

In [1]:
import chipwhisperer as cw


# Firmware Elena
#target = cw.target(None, cw.targets.CW305, bsfile="/home/user/vivado_projects_ana/project_remoterst_activity/cw305_remoterst_activity.bit", slurp = False)
#target = cw.target(None, cw.targets.CW305, bsfile="/home/user/chipwhisperer/cw305_remote_signals_debug.bit", slurp = False)

target = cw.target(None, cw.targets.CW305, bsfile="/home/user/vivado_projects_ana/project_remoterst_activity/project_remote_reset_activity.bit", slurp=False)

#target = cw.target(None, cw.targets.CW305, bsfile="/home/user/DVFS/cw305_nocnt_cg.bit", slurp=False)


#ecg_debug_normal_2.bit
#cw305_remote_reset.bit

# Firmware nuevo
#target = cw.target(None, cw.targets.CW305, bsfile="/home/user/Escritorio/cw305_remote_signals_final_debug2.bit", slurp = False)
#target = cw.target(None, cw.targets.CW305, bsfile="/home/user/Descargas/ecg.bit", slurp = False)

#Firmware DVFS
#target = cw.target(None, cw.targets.CW305, bsfile="/home/user/DVFS/ecg.bit", slurp = False)

See https://chipwhisperer.readthedocs.io/en/latest/api.html#firmware-update


In [2]:
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
target.pll.pll_outfreq_set(100E6, 1)
#target.pll.pll_outfreq_set(10E6, 1)

In [3]:
target.fpga_write(0,[1])

In [4]:
target.fpga_write(0,[3])

In [5]:
target.fpga_write(0,[2])

In [6]:
target.fpga_write(0,[3])

In [110]:
print(target.fpga_read(0, 1))

bytearray(b'\x1f')


# DVFS loop

In [7]:
import subprocess
import signal
import time

ECG_dir = '/home/user/ecg_20240927/implementation_v2/hw/C/'
command_send_high = ECG_dir + 'bin/configure_and_send_stoppable'
command_receive_high = ECG_dir + 'bin/receive_fits'

command_send_low = ECG_dir + 'bin/configure_and_send_stoppable_half'
command_receive_low = ECG_dir + 'bin/receive_fits_half'
filter_name = ECG_dir + 'filter/tfilter2.i16.2.dc.txt'

#################
db_name = ECG_dir + 'data/LK27_walkbefore_run_cut.txt'
################

filter_order = '32'
uart = '/dev/ttyUSB0'
wait_time = 10 # If the value is 10 or lower, it stops working properly
wait_time_threshold = 2 
current_config = "LOW"
prev_config = "LOW"
starting_line = 0        
pipe_in = open(uart, "rb")

# Reset the FPGA to start clean
def reset_fpga():
    target.fpga_write(0,[1])
    time.sleep(1)
    target.fpga_write(0,[3])
    time.sleep(1)
    target.fpga_write(0,[2])
    time.sleep(1)
    target.fpga_write(0,[3])
    time.sleep(1)
    
    
def start_receiver(config, pipe_in, of_receive):
    if config == "LOW":
        command_receive = command_receive_low
    else:
        command_receive = command_receive_high
    
    
    process = subprocess.Popen([command_receive], stdin = pipe_in, stdout = of_receive, stderr = subprocess.STDOUT, text = True)
    return process
    
reset_fpga() 

# CONFIGURACION INICIAL
target.vccint_set(0.8)
target.pll.pll_outfreq_set(50E6, 1)
time.sleep(1)

###############
# Open the output file and start the receiver process
of_receive = open("FITS/LK27_FITS_02.txt", "wb")
##############

#Por defecto arranca en LOW
receive_process = start_receiver("LOW", pipe_in, of_receive)
time.sleep(1)

# Run for 2 minutes
start_time = time.time()
while(time.time() - start_time < 120):  
    
    if current_config == "LOW":
        command_send = command_send_low
    else:
        command_send = command_send_high
    
    # Start sending ECG data from where we left off
    if starting_line == 0:
        cmd = [command_send, filter_order, filter_name, db_name, uart]
    else:
        cmd = [command_send, filter_order, filter_name, db_name, uart, str(starting_line)]

    # Starts configure_and_send
    send_process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Wait before reading the threshold for the first time
    time.sleep(wait_time)
    
    config_changed = False
    
    # Keep reading the threhold every 2 seconds until config changes or time is up
    while config_changed == False:
        
        time.sleep(wait_time_threshold)
        
        # Stop if 2 minutes have passed
        if time.time() - start_time >= 120:
            send_process.send_signal(signal.SIGINT)
            break
        
        # Read the activity level from FPGA
        threshold_raw = target.fpga_read(0, 1)
        threshold = (threshold_raw[0] >> 4) & 0x03
        print("Threshold: ", threshold)
            
        # Determine if activity is LOW or HIGH    
        if threshold in [0, 1]:
            current_config = "LOW"
        else:
            current_config = "HIGH"
            
        # If config changed, stop sending, reset and reconfigure
        if current_config != prev_config:
            send_process.send_signal(signal.SIGINT)
            _, output = send_process.communicate()

            for line in output.split('\n'):
                if line.startswith("Line:"):
                    starting_line = int(line.split(":")[1].strip())


            print("Starting line: ", starting_line)
            
            #Parar receiver actual
            receive_process.send_signal(signal.SIGINT)
            receive_process.wait()
        
            reset_fpga()
            print("Board Reset Completed")

            # Apply LOW configuration: 50MHz-0.8V
            if current_config == "LOW":
                print("[LOW]")
                target.vccint_set(0.8)
                target.pll.pll_outfreq_set(50E6, 1)
                
            # Apply HIGH configuration: 100MHz-1V
            if current_config == "HIGH":
                print("[HIGH]")
                target.vccint_set(1.0)
                target.pll.pll_outfreq_set(100E6, 1)
                
            #Se vuelve a lanzar el receiver     
            receive_process = start_receiver(current_config, pipe_in, of_receive)
            time.sleep(1)

            prev_config = current_config
            config_changed = True

# Stop the receiver and close everything
receive_process.send_signal(signal.SIGINT)
receive_process.wait()

pipe_in.close() 

of_receive.close()

print("Process finished.")

    

Threshold:  0
Threshold:  0
Threshold:  0
Threshold:  0
Threshold:  0
Threshold:  3
Starting line:  14772
Board Reset Completed
[HIGH]
Threshold:  1
Starting line:  22967
Board Reset Completed
[LOW]
Threshold:  0
Threshold:  0
Threshold:  0
Threshold:  0
Threshold:  3
Starting line:  35947
Board Reset Completed
[HIGH]
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  3
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  1
Starting line:  74380
Board Reset Completed
[LOW]
Process finished.


# DVFS loop (Forced HIGH Configuration)

In [10]:
import subprocess
import signal
import time

ECG_dir = '/home/user/ecg_20240927/implementation_v2/hw/C/'
command_send_high = ECG_dir + 'bin/configure_and_send_stoppable'
command_receive_high = ECG_dir + 'bin/receive_fits'

filter_name = ECG_dir + 'filter/tfilter2.i16.2.dc.txt'

###############
db_name = ECG_dir + 'data/mlii_arritmia_cut.txt'
##############

filter_order = '32'
uart = '/dev/ttyUSB0'
wait_time = 10 # If the value is 10 or lower, it stops working properly
wait_time_threshold = 2 
starting_line = 0        
pipe_in = open(uart, "rb")

# Reset the FPGA to start clean
def reset_fpga():
    target.fpga_write(0,[1])
    time.sleep(1)
    target.fpga_write(0,[3])
    time.sleep(1)
    target.fpga_write(0,[2])
    time.sleep(1)
    target.fpga_write(0,[3])
    time.sleep(1)
    
reset_fpga()    

######################
# Open the output file and start the receiver process
of_receive = open("FITS/CG/MLII_FITS.txt", "wb")
#####################

#Por defecto arranca en HIGH
receive_process = subprocess.Popen([command_receive_high], stdin = pipe_in, stdout = of_receive, stderr = subprocess.STDOUT, text = True)
time.sleep(1)

# Run for 2 minutes
start_time = time.time()
while(time.time() - start_time < 120):  
    
    # Start sending ECG data from where we left off
    if starting_line == 0:
        cmd = [command_send_high, filter_order, filter_name, db_name, uart]
    else:
        cmd = [command_send_high, filter_order, filter_name, db_name, uart, str(starting_line)]

    # Starts configure_and_send
    send_process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    # Wait before reading the threshold for the first time
    time.sleep(wait_time)
    
    config_changed = False
    # Keep reading the threhold every 2 seconds until config changes or time is up
    while config_changed == False:
        
        time.sleep(wait_time_threshold)
        
        # Stop if 2 minutes have passed
        if time.time() - start_time >= 120:
            send_process.send_signal(signal.SIGINT)
            break
        
        # Read the activity level from FPGA
        threshold_raw = target.fpga_read(0, 1)
        threshold = (threshold_raw[0] >> 4) & 0x03
        print("Threshold: ", threshold)

# Stop the receiver and close everything
receive_process.send_signal(signal.SIGINT)
receive_process.wait()

pipe_in.close() 

of_receive.close()

print("Process finished.")
    

Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  2
Threshold:  1
Threshold:  1
Threshold:  1
Threshold:  0
Threshold:  0
Threshold:  0
Process finished.


In [18]:
receive_process.send_signal(signal.SIGINT)
receive_process.wait()

pipe_in.close() 

of_receive.close()

In [1]:
a = []
for i in range(100):
    a.append(target.fpga_read(0, 1))

NameError: name 'target' is not defined

In [18]:
import numpy as np
np.unique(a)

array([63], dtype=uint8)